# Asia C Failure Diagnosis

Postmortem of Universe C (HK/JP) locked pairs under the frozen trad-z / OLS / 1d stack.
Not a new hypothesis — does not change `KEEP_PAIR_IDS` or the star stack.

**Pairs:** `1398.HK|0939.HK`, `1288.HK|3328.HK`, `8306.T|8316.T`  
**Default window:** research IS only (`date <= RESEARCH_IS_END`). Set `USE_OUT_OF_SAMPLE_DATA = True` to include sealed OOS.  
**Helpers:** `04_backtest/s2_coint/diagnosis.py`

### What strategy this notebook uses

Simulates the **H-001 baseline only** via `simulate_pair_baseline` (same path as H-001 IS diagnostics):

| In use | Not in use |
|--------|------------|
| Trad-z entry/exit (`ENTRY_Z`, `EXIT_Z`) | Full `engine.simulate_pair` / `run_s2_backtest` |
| OLS hedge already on the panel (matches H-003 `HEDGE_STAR=ols`) | Kalman β |
| 1d bars (matches H-002 `BAR_STAR=1d`) | 1H |
| Beta-sized legs + IBKR HK/JP costs | ADF / variance-jump **trade** kill-switch (H-003) |
| Timing: signal close `t` → fill open `t+1` | HL gate, trend filter, overlap rules, alternate exits/sizing (H-007+) |

**ADF:** rolling `adf_pvalue` appears in the scorecard and Plotly panels as a **health diagnostic only** (`COINT_PVALUE=0.05` for summary %). It does **not** block entries or flatten positions here — there is no ADF cutoff in the traded rules of this notebook.

## 0. Imports & Config

In [1]:
from __future__ import annotations

import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.diagnosis import (
    check_fill_timing,
    failure_scorecard,
    plotly_pair_diagnosis,
    print_extreme_trades,
    simulate_and_enrich_panel,
    slice_panel_window,
)
from backtest.s2_coint.research import (
    FROZEN_C_PAIRS,
    RESEARCH_IS_END_C,
    load_universe_c_panels,
    panel_paths,
)
from strategies.s2_coint.baseline import ENTRY_Z, EXIT_Z

USE_OUT_OF_SAMPLE_DATA = False
Z_WINDOW = 60
RESEARCH_IS_END = RESEARCH_IS_END_C
PAIRS = list(FROZEN_C_PAIRS)
N_EXTREME = 3

print("ROOT", ROOT)
print("PAIRS", PAIRS)
print("RESEARCH_IS_END", RESEARCH_IS_END)
print("USE_OUT_OF_SAMPLE_DATA", USE_OUT_OF_SAMPLE_DATA)
print("ENTRY_Z", ENTRY_Z, "EXIT_Z", EXIT_Z, "Z_WINDOW", Z_WINDOW)

ROOT c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
PAIRS ['1398.HK|0939.HK', '1288.HK|3328.HK', '8306.T|8316.T']
RESEARCH_IS_END 2021-12-31
USE_OUT_OF_SAMPLE_DATA False
ENTRY_Z 2.0 EXIT_Z 0.0 Z_WINDOW 60


## 1. Failure-mode legend

| Section | Failure looks like |
|---------|-------------------|
| **Opportunity** | Few entries / round-trips; `% days \|z\| > ENTRY_Z` near 0 (flat z, almost never crosses bands). |
| **Costs** | `ann_sharpe_gross` clearly positive while `ann_sharpe_net` ≤ 0, or `cost_bps_year` large vs edge. |
| **No edge** | Both gross and net Sharpe ≤ 0 (or near 0 with noisy PnL). |
| **Noise / horizon** | Many trades, median hold ≪ median half-life, expectancy near zero. |
| **Coint health** | High median/last ADF p, low `% ADF < 0.05`, or large `beta_std` (unstable hedge). |
| **Fill timing** | Any `all_ok == False` in the timing table (same-bar close fill or missing open). |

**Gross vs net:** Gross Sharpe ignores commissions/spread/slippage; net includes them — if gross ≫ 0 and net ≤ 0, costs ate the edge; if both ≤ 0, there was no edge even before costs.

## 2. Load panels

Requires `s2_panel_C_1d_train.parquet` (and `s2_panel_C_1d_full.parquet` when `USE_OUT_OF_SAMPLE_DATA` is True).
Rebuild: run `H-001_universes.ipynb` lock step, then `01_data/data_files/s2_coint/s2_pair_panel.ipynb`.

In [2]:
train_path, full_path = panel_paths("1d", root=ROOT)
print("train", train_path, "exists", os.path.isfile(train_path))
print("full ", full_path, "exists", os.path.isfile(full_path))

train, full = load_universe_c_panels("1d", PAIRS, root=ROOT)
source = full if USE_OUT_OF_SAMPLE_DATA else train
panel = slice_panel_window(source, RESEARCH_IS_END, USE_OUT_OF_SAMPLE_DATA)
panel = panel.loc[panel["pair_id"].isin(PAIRS)].copy()

print("rows", len(panel), "pairs", sorted(panel["pair_id"].unique().tolist()))
print("date range", panel["date"].min(), "→", panel["date"].max())
if USE_OUT_OF_SAMPLE_DATA:
    n_oos = int((panel["date"] > pd.Timestamp(RESEARCH_IS_END)).sum())
    print("OOS rows", n_oos)

train c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s2_coint\s2_panel_C_1d_train.parquet exists True
full  c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s2_coint\s2_panel_C_1d_full.parquet exists True
rows 10592 pairs ['1288.HK|3328.HK', '1398.HK|0939.HK', '8306.T|8316.T']
date range 2005-09-29 00:00:00 → 2021-12-31 00:00:00


## 3. Failure scorecard

Per-pair opportunity, net/gross edge, cost drag, and cointegration health on the loaded window.

In [3]:
scorecard = failure_scorecard(
    panel,
    entry_z=ENTRY_Z,
    exit_z=EXIT_Z,
    gross=True,
)
scorecard

,pair_id,n_entries,n_round_trips,pct_days_abs_z_gt_entry,median_hold_bars,median_half_life,ann_sharpe_net,ann_sharpe_gross,max_drawdown,cost_bps_year,median_adf_p,last_adf_p,pct_adf_lt_threshold,beta_std,n_days
0,1288.HK|3328.HK,43,42,12.619048,28.0,22.297215,-0.480647,0.050374,-0.405549,439.306931,0.302422,0.087961,0.113832,0.451250,2828
1,1398.HK|0939.HK,54,54,11.874272,24.0,16.600571,-0.476618,0.022838,-0.497258,421.615385,0.194580,0.168227,0.293157,0.181290,3744
2,8306.T|8316.T,68,67,12.088505,24.0,15.974046,-0.082521,0.238706,-0.262648,271.210762,0.231685,0.325532,0.181559,0.239519,4014


### 3.1 Optional IS vs OOS split

Runs only when `USE_OUT_OF_SAMPLE_DATA` is True.

In [4]:
if USE_OUT_OF_SAMPLE_DATA:
    is_panel = panel.loc[panel["date"] <= pd.Timestamp(RESEARCH_IS_END)].copy()
    oos_panel = panel.loc[panel["date"] > pd.Timestamp(RESEARCH_IS_END)].copy()
    print("=== Research IS ===")
    display(failure_scorecard(is_panel, entry_z=ENTRY_Z, exit_z=EXIT_Z, gross=True))
    print("=== Sealed OOS ===")
    display(failure_scorecard(oos_panel, entry_z=ENTRY_Z, exit_z=EXIT_Z, gross=True))
else:
    print("USE_OUT_OF_SAMPLE_DATA=False — skip IS/OOS split tables.")

USE_OUT_OF_SAMPLE_DATA=False — skip IS/OOS split tables.


## 4. Extreme trades

Best/worst round-trips by **net** `pnl_pct` (percent of that pair’s capital, costs included). Use the printed dates to zoom the Plotly charts.

In [5]:
trades_enriched, returns_by_pair = simulate_and_enrich_panel(
    panel, entry_z=ENTRY_Z, exit_z=EXIT_Z
)
print("n round-trips", len(trades_enriched))
print_extreme_trades(trades_enriched, n=N_EXTREME)
trades_enriched.sort_values("pnl_pct", ascending=False).head(10)

n round-trips 163
=== Best 3 trades by net pnl_pct ===
        pair_id side_label entry_date  exit_date  pnl_pct   z_entry
  8306.T|8316.T      short 2008-10-21 2008-10-28 7.856465  2.191998
1288.HK|3328.HK       long 2020-07-03 2020-07-07 4.439397 -2.039487
1398.HK|0939.HK      short 2011-11-10 2011-11-22 3.610313  2.171692

=== Worst 3 trades by net pnl_pct ===
        pair_id side_label entry_date  exit_date   pnl_pct   z_entry
1288.HK|3328.HK       long 2018-09-21 2019-01-25 -8.290762 -2.555681
1288.HK|3328.HK       long 2014-07-29 2014-12-24 -7.882489 -2.460228
1398.HK|0939.HK       long 2011-08-12 2011-10-26 -7.548801 -2.179403


,pair_id,side,side_label,entry_date,exit_date,hold_bars,entry_cost_bps,exit_cost_bps,signal_date,z_entry,spread_entry,pnl_pct
106,8306.T|8316.T,-1,short,2008-10-21,2008-10-28,5,32.0,32.0,2008-10-20,2.191998,0.110153,7.856465
35,1288.HK|3328.HK,1,long,2020-07-03,2020-07-07,2,58.0,58.0,2020-07-02,-2.039487,-0.002078,4.439397
57,1398.HK|0939.HK,-1,short,2011-11-10,2011-11-22,8,58.0,58.0,2011-11-09,2.171692,0.045555,3.610313
100,8306.T|8316.T,1,long,2007-09-28,2007-10-03,3,32.0,32.0,2007-09-27,-2.860112,-0.090065,3.348647
44,1398.HK|0939.HK,1,long,2008-08-13,2008-08-27,10,58.0,58.0,2008-08-12,-3.208676,-0.016091,2.920587
108,8306.T|8316.T,-1,short,2009-03-18,2009-03-24,3,32.0,32.0,2009-03-17,2.053913,-0.011097,2.548353
102,8306.T|8316.T,1,long,2007-11-28,2007-12-05,5,32.0,32.0,2007-11-27,-2.063334,-0.064527,2.214585
28,1288.HK|3328.HK,-1,short,2019-02-04,2019-05-07,59,58.0,58.0,2019-02-01,2.085122,-0.098535,2.205880
110,8306.T|8316.T,1,long,2009-07-16,2009-09-18,44,32.0,32.0,2009-07-15,-2.292337,-0.015074,2.185324
24,1288.HK|3328.HK,1,long,2018-03-12,2018-03-14,2,58.0,58.0,2018-03-09,-2.078922,0.021121,1.710132


## 5. Interactive plots

Per pair: spread with Bollinger bands at `± ENTRY_Z · σ` over `Z_WINDOW` (same standardization as trad-z), green/red entry/exit markers, rolling ADF p, beta, cumulative net PnL.
When `USE_OUT_OF_SAMPLE_DATA` is True, a vertical line marks `RESEARCH_IS_END`.

In [6]:
figures = []
for pair_id in PAIRS:
    g = panel.loc[panel["pair_id"] == pair_id].copy()
    t = trades_enriched.loc[trades_enriched["pair_id"] == pair_id].copy()
    fig = plotly_pair_diagnosis(
        g,
        t,
        entry_z=ENTRY_Z,
        z_window=Z_WINDOW,
        pair_returns=returns_by_pair.get(pair_id),
        is_end=RESEARCH_IS_END if USE_OUT_OF_SAMPLE_DATA else None,
        title=f"{pair_id} · {'IS+OOS' if USE_OUT_OF_SAMPLE_DATA else 'research IS'}",
    )
    figures.append(fig)
    fig.show()
print("plotted", len(figures), "pairs")

plotted 3 pairs


## 6. Fill-timing check

**Pass:** decision uses finite z on close of signal bar `t`; fill is open of the next panel session (`entry_date`); signal date ≠ fill date (not same-bar close fill).

**Fail:** any `all_ok == False` — inspect that trade’s `signal_date` / `entry_date` / open columns.

In [7]:
timing = check_fill_timing(trades_enriched, panel)
n_fail = int((~timing["all_ok"]).sum()) if not timing.empty else 0
print("timing rows", len(timing), "failures", n_fail)
timing

timing rows 163 failures 0


,pair_id,entry_date,signal_date,ok_signal_before_fill,ok_fill_has_open,ok_not_same_bar_close_fill,ok_z_finite_on_signal,all_ok
0,1288.HK|3328.HK,2012-03-20,2012-03-16,True,True,True,True,True
1,1288.HK|3328.HK,2012-07-05,2012-07-04,True,True,True,True,True
2,1288.HK|3328.HK,2012-08-29,2012-08-28,True,True,True,True,True
3,1288.HK|3328.HK,2012-11-01,2012-10-31,True,True,True,True,True
4,1288.HK|3328.HK,2013-03-18,2013-03-15,True,True,True,True,True
...,...,...,...,...,...,...,...,...
158,8306.T|8316.T,2021-01-08,2021-01-07,True,True,True,True,True
159,8306.T|8316.T,2021-02-04,2021-02-03,True,True,True,True,True
160,8306.T|8316.T,2021-05-14,2021-05-13,True,True,True,True,True
161,8306.T|8316.T,2021-07-12,2021-07-09,True,True,True,True,True
